In [1]:
import json
from datetime import datetime, timedelta, timezone
from pathlib import Path

import pandas as pd
import requests

BASE = "https://api.carbonintensity.org.uk"
HEADERS = {"Accept": "application/json"}
LONDON_REGION_ID = 13


def get_json(path):
    """Call one API endpoint and return (status_code, parsed JSON or error text)."""
    r = requests.get(f"{BASE}{path}", headers=HEADERS, timeout=30)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text


def iso(dt):
    """Format a datetime the way the API expects: 2026-10-16T12:00Z"""
    return dt.strftime("%Y-%m-%dT%H:%MZ")


now = datetime.now(timezone.utc)
print("Now (UTC):", iso(now))

Now (UTC): 2026-10-16T12:02Z


In [10]:
status, payload = get_json("/intensity")
print(status)
print(json.dumps(payload, indent=2))

200
{
  "data": [
    {
      "from": "2026-10-16T11:30Z",
      "to": "2026-10-16T12:00Z",
      "intensity": {
        "forecast": 73,
        "actual": 74,
        "index": "low"
      }
    }
  ]
}


In [3]:
start = now - timedelta(days=7)
status, payload = get_json(f"/intensity/{iso(start)}/{iso(now)}")
print(status)

nat = pd.json_normalize(payload["data"])
nat["from"] = pd.to_datetime(nat["from"], utc=True)
nat["to"] = pd.to_datetime(nat["to"], utc=True)

print(nat.dtypes, "\n")
print("Rows:", len(nat), "(expect about 7 x 48 = 336)")
print("Nulls per column:\n", nat.isna().sum(), "\n")
nat.tail(10)

200
from                  datetime64[us, UTC]
to                    datetime64[us, UTC]
intensity.forecast                  int64
intensity.actual                  float64
intensity.index                       str
dtype: object 

Rows: 336 (expect about 7 x 48 = 336)
Nulls per column:
 from                  0
to                    0
intensity.forecast    0
intensity.actual      1
intensity.index       0
dtype: int64 



,from,to,intensity.forecast,intensity.actual,intensity.index
326,2026-10-16 07:00:00+00:00,2026-10-16 07:30:00+00:00,161,167.0,moderate
327,2026-10-16 07:30:00+00:00,2026-10-16 08:00:00+00:00,156,157.0,moderate
328,2026-10-16 08:00:00+00:00,2026-10-16 08:30:00+00:00,143,141.0,moderate
329,2026-10-16 08:30:00+00:00,2026-10-16 09:00:00+00:00,130,116.0,moderate
330,2026-10-16 09:00:00+00:00,2026-10-16 09:30:00+00:00,117,95.0,moderate
331,2026-10-16 09:30:00+00:00,2026-10-16 10:00:00+00:00,105,85.0,low
332,2026-10-16 10:00:00+00:00,2026-10-16 10:30:00+00:00,86,80.0,low
333,2026-10-16 10:30:00+00:00,2026-10-16 11:00:00+00:00,74,76.0,low
334,2026-10-16 11:00:00+00:00,2026-10-16 11:30:00+00:00,69,74.0,low
335,2026-10-16 11:30:00+00:00,2026-10-16 12:00:00+00:00,73,NaN,low


In [4]:
for days in [14, 31, 60]:
    s = now - timedelta(days=days)
    status, payload = get_json(f"/intensity/{iso(s)}/{iso(now)}")
    if status == 200 and isinstance(payload, dict) and "data" in payload:
        print(f"{days} days -> OK, {len(payload['data'])} rows")
    else:
        print(f"{days} days -> status {status}: {str(payload)[:200]}")

14 days -> OK, 672 rows
31 days -> status 400: {'error': {'code': '400 Bad Request', 'message': 'The date range you have specified is greater than 31 days. Please select a smaller date range.'}}
60 days -> status 400: {'error': {'code': '400 Bad Request', 'message': 'The date range you have specified is greater than 31 days. Please select a smaller date range.'}}


In [5]:
for d in ["2017-09-26", "2018-01-01", "2018-05-10", "2019-01-01"]:
    s = datetime.fromisoformat(d).replace(tzinfo=timezone.utc)
    e = s + timedelta(days=1)
    status, payload = get_json(f"/intensity/{iso(s)}/{iso(e)}")
    rows = len(payload["data"]) if status == 200 and "data" in payload else 0
    print(d, "-> status", status, "| rows:", rows)

2017-09-26 -> status 200 | rows: 49
2018-01-01 -> status 200 | rows: 49
2018-05-10 -> status 200 | rows: 49
2019-01-01 -> status 200 | rows: 49


In [6]:
status, payload = get_json("/generation")
print(status)
pd.json_normalize(payload["data"]["generationmix"])

200


,fuel,perc
0,biomass,9.3
1,coal,0.0
2,imports,14.2
3,gas,12.0
4,nuclear,14.1
5,other,0.0
6,hydro,0.0
7,solar,18.9
8,wind,31.5


In [7]:
status, payload = get_json(f"/regional/regionid/{LONDON_REGION_ID}")
print(status)
print(json.dumps(payload, indent=2)[:1500])

200
{
  "data": [
    {
      "regionid": 13,
      "dnoregion": "UKPN London",
      "shortname": "London",
      "data": [
        {
          "from": "2026-10-16T11:30Z",
          "to": "2026-10-16T12:00Z",
          "intensity": {
            "forecast": 75,
            "index": "low"
          },
          "generationmix": [
            {
              "fuel": "biomass",
              "perc": 3.5
            },
            {
              "fuel": "coal",
              "perc": 0
            },
            {
              "fuel": "imports",
              "perc": 34.8
            },
            {
              "fuel": "gas",
              "perc": 6.7
            },
            {
              "fuel": "nuclear",
              "perc": 11.5
            },
            {
              "fuel": "other",
              "perc": 0
            },
            {
              "fuel": "hydro",
              "perc": 0.1
            },
            {
              "fuel": "solar",
              "perc

In [8]:
status, payload = get_json(
    f"/regional/intensity/{iso(start)}/{iso(now)}/regionid/{LONDON_REGION_ID}"
)
print(status)

ldn = pd.json_normalize(payload["data"]["data"])
print("Columns:", [c for c in ldn.columns if c.startswith("intensity")])
print("Rows:", len(ldn))
ldn[["from", "to"] + [c for c in ldn.columns if c.startswith("intensity")]].tail(10)

200
Columns: ['intensity.forecast', 'intensity.index']
Rows: 336


,from,to,intensity.forecast,intensity.index
326,2026-10-16T07:00Z,2026-10-16T07:30Z,202,high
327,2026-10-16T07:30Z,2026-10-16T08:00Z,166,moderate
328,2026-10-16T08:00Z,2026-10-16T08:30Z,131,moderate
329,2026-10-16T08:30Z,2026-10-16T09:00Z,114,moderate
330,2026-10-16T09:00Z,2026-10-16T09:30Z,97,moderate
331,2026-10-16T09:30Z,2026-10-16T10:00Z,91,moderate
332,2026-10-16T10:00Z,2026-10-16T10:30Z,81,low
333,2026-10-16T10:30Z,2026-10-16T11:00Z,76,low
334,2026-10-16T11:00Z,2026-10-16T11:30Z,76,low
335,2026-10-16T11:30Z,2026-10-16T12:00Z,75,low


In [9]:
status, payload = get_json(f"/intensity/{iso(now)}/fw24h")
print(status)

fw = pd.json_normalize(payload["data"])
print("Rows:", len(fw), "(expect about 48)")

snap_dir = Path("../data/raw/neso_fw24h_snapshots")
snap_dir.mkdir(parents=True, exist_ok=True)
snap_file = snap_dir / f"fw24h_{now.strftime('%Y%m%dT%H%M')}.json"
snap_file.write_text(json.dumps(payload))
print("Saved:", snap_file)
fw.head()

200
Rows: 48 (expect about 48)
Saved: ..\data\raw\neso_fw24h_snapshots\fw24h_20261016T1202.json


,from,to,intensity.forecast,intensity.actual,intensity.index
0,2026-10-16T12:00Z,2026-10-16T12:30Z,69,None,low
1,2026-10-16T12:30Z,2026-10-16T13:00Z,70,None,low
2,2026-10-16T13:00Z,2026-10-16T13:30Z,65,None,low
3,2026-10-16T13:30Z,2026-10-16T14:00Z,59,None,low
4,2026-10-16T14:00Z,2026-10-16T14:30Z,61,None,low


## Findings

1. **Regional actuals:** not available (London returns forecast and index only). Target = national GB intensity. London differs materially from national, so framing is GB-wide.
2. **Missing actuals (national, 7 days):** 1 of 336, the most recent period (publication lag). Recheck over full history.
3. **Max range per request:** 14 days OK, 31 rejected. Backfill in 30-day chunks.
4. **Historical forecast = true day-ahead?** Pending. Snapshot saved 2026-10-16 12:02 UTC; compare tomorrow.

Other observations:
- Each daily request returns 49 rows (boundaries inclusive), so chunks overlap: deduplicate on `from`.
- Early dates (2017) return rows; need to check whether values are null.
- NESO forecast underestimated the morning solar drop (09:00: 117 forecast vs 95 actual). Solar/weather features may help.
- Generation mix explains intensity but isn't known in advance: only lagged mix allowed as a feature (leakage risk).